In [1]:
# KandaRakshak prototype

import numpy as np
import trimesh

# --- COLOR PALETTE (RGBA) ---
COLOR_WOOD = [185, 125, 68, 255]  # Crate Frame & Slats
COLOR_PLASTIC_DARK = [35, 35, 35, 255]  # Base Manifold & Outer Enclosure
COLOR_HEADER_METAL = [45, 48, 52, 255]  # Dark Solar Chimney Hood
COLOR_STAINLESS_STEEL = [195, 200, 205, 255]  # Metallic Stainless Steel
COLOR_INTAKE_MESH = [160, 165, 170, 210]  # Wire Mesh
COLOR_ZEOLITE = [235, 235, 225, 255]
COLOR_BIOCHAR = [15, 15, 15, 255]
COLOR_VOC_PAD = [200, 150, 95, 255]
COLOR_ONION = [225, 140, 45, 255]
COLOR_ONION_STEM = [170, 100, 25, 255]

# --- GEOMETRY HELPER FUNCTIONS ---
def box(ext, pos, color):
    m = trimesh.creation.box(extents=ext)
    m.apply_translation(pos)
    m.visual.face_colors = color
    return m

def cyl(r, h, pos, color):
    m = trimesh.creation.cylinder(radius=r, height=h)
    m.apply_translation(pos)
    m.visual.face_colors = color
    return m

def cone(r, h, pos, color):
    m = trimesh.creation.cone(radius=r, height=h)
    m.apply_translation(pos)
    m.visual.face_colors = color
    return m

def perforated_plate(extents, pos, hole_r=0.012, grid_size=(8, 8), color=COLOR_STAINLESS_STEEL):
    base_plate = trimesh.creation.box(extents=extents)
    x_coords = np.linspace(-extents[0] * 0.4, extents[0] * 0.4, grid_size[0])
    y_coords = np.linspace(-extents[1] * 0.4, extents[1] * 0.4, grid_size[1])

    holes = []
    for x in x_coords:
        for y in y_coords:
            h = trimesh.creation.cylinder(radius=hole_r, height=extents[2] * 1.5)
            h.apply_translation([x, y, 0])
            holes.append(h)

    if holes:
        all_holes = trimesh.util.concatenate(holes)
        try:
            plate = base_plate.difference(all_holes)
        except Exception:
            plate = base_plate
    else:
        plate = base_plate

    plate.apply_translation(pos)
    plate.visual.face_colors = color
    return plate

def onion(r, pos):
    b = trimesh.creation.icosphere(radius=r, subdivisions=2)
    b.apply_transform(np.diag([1.0, 1.0, 0.75, 1.0]))
    b.visual.face_colors = COLOR_ONION

    t = trimesh.creation.cone(radius=r * 0.25, height=r * 0.5)
    t.apply_translation([0, 0, r * 0.65])
    t.visual.face_colors = COLOR_ONION_STEM

    o = trimesh.util.concatenate([b, t])
    o.apply_translation(pos)
    return o

# Master list to store all generated meshes
all_meshes = []

# --- 1. WOODEN BASE CRATE PIECES ---
all_meshes.append(box([1.22, 0.08, 0.04], [0, -0.54, 0.02], COLOR_WOOD))
all_meshes.append(box([1.22, 0.08, 0.04], [0, 0.54, 0.02], COLOR_WOOD))
all_meshes.append(box([0.08, 1.00, 0.04], [-0.54, 0, 0.02], COLOR_WOOD))
all_meshes.append(box([0.08, 1.00, 0.04], [0.54, 0, 0.02], COLOR_WOOD))

# --- 2. FULL-COVERAGE SORBENT INLET ---
inlet_z_start = 0.04
inlet_height = 0.08
z_inlet_mid = inlet_z_start + (inlet_height / 2.0)
wall_thick = 0.03
full_w = 1.20

# Continuous solid bottom base plate to bridge/bind all lower components
all_meshes.append(
    box([full_w, full_w, 0.015], [0, 0, inlet_z_start + 0.0075], COLOR_PLASTIC_DARK)
)

win_w = 0.38
win_h = 0.045
mesh_thick = 0.005

for x_off in [-0.28, 0.28]:
    all_meshes.append(box([win_w, mesh_thick, win_h], [x_off, -full_w / 2.0, z_inlet_mid], COLOR_INTAKE_MESH))
    all_meshes.append(box([win_w, mesh_thick, win_h], [x_off, full_w / 2.0, z_inlet_mid], COLOR_INTAKE_MESH))

for y_off in [-0.28, 0.28]:
    all_meshes.append(box([mesh_thick, win_w, win_h], [-full_w / 2.0, y_off, z_inlet_mid], COLOR_INTAKE_MESH))
    all_meshes.append(box([mesh_thick, win_w, win_h], [full_w / 2.0, y_off, z_inlet_mid], COLOR_INTAKE_MESH))

for x in [-0.585, 0.0, 0.585]:
    for y in [-0.585, 0.585]:
        all_meshes.append(box([0.10, wall_thick, win_h], [x, y, z_inlet_mid], COLOR_PLASTIC_DARK))

for y in [-0.585, 0.0, 0.585]:
    for x in [-0.585, 0.585]:
        all_meshes.append(box([wall_thick, 0.10, win_h], [x, y, z_inlet_mid], COLOR_PLASTIC_DARK))

# --- 3. PERFORATED MESH LAYER ---
z_sub_mesh = inlet_z_start + inlet_height
Y_C = -0.02
Y_L = 1.12
all_meshes.append(perforated_plate([1.14, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_sub_mesh]))

# --- 4. CRATE FRAME & SLATS ---
for x, y in [(-0.575, -0.575), (-0.575, 0.575), (0.575, -0.575), (0.575, 0.575)]:
    all_meshes.append(box([0.05, 0.05, 0.96], [x, y, z_sub_mesh + 0.48], COLOR_WOOD))

slat_z_positions = np.linspace(z_sub_mesh + 0.05, z_sub_mesh + 0.90, 10)
for z_slat in slat_z_positions:
    all_meshes.append(box([0.02, 1.10, 0.085], [-0.575, 0, z_slat], COLOR_WOOD))
    all_meshes.append(box([0.02, 1.10, 0.085], [0.575, 0, z_slat], COLOR_WOOD))
    all_meshes.append(box([1.10, 0.02, 0.085], [0, 0.575, z_slat], COLOR_WOOD))

# --- 5. CASSETTE LAYERS ---
z_layer1 = z_sub_mesh + 0.025
all_meshes.append(box([1.08, Y_L, 0.045], [0, Y_C, z_layer1], COLOR_ZEOLITE))

z_mesh1 = z_layer1 + 0.0255
all_meshes.append(perforated_plate([1.12, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_mesh1]))

z_layer2 = z_mesh1 + 0.025
all_meshes.append(box([1.08, Y_L, 0.045], [0, Y_C, z_layer2], COLOR_BIOCHAR))

z_mesh2 = z_layer2 + 0.0255
all_meshes.append(perforated_plate([1.12, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_mesh2]))

z_layer3 = z_mesh2 + 0.018
all_meshes.append(box([1.08, Y_L, 0.030], [0, Y_C, z_layer3], COLOR_VOC_PAD))

z_mesh_top = z_layer3 + 0.018
all_meshes.append(perforated_plate([1.12, Y_L + 0.02, 0.006], [0, Y_C - 0.01, z_mesh_top]))

# --- 6. CROP TIERS ---
shelf_heights = [z_mesh_top + 0.05, z_mesh_top + 0.23, z_mesh_top + 0.41, z_mesh_top + 0.59]
for z_sh in shelf_heights:
    all_meshes.append(perforated_plate([1.10, 1.10, 0.008], [0, 0, z_sh], hole_r=0.015, grid_size=(10, 10)))
    for xi in range(6):
        for yi in range(6):
            xp = -0.42 + (xi * 0.168)
            yp = -0.42 + (yi * 0.168)
            all_meshes.append(onion(0.040, [xp, yp, z_sh + 0.038]))

# --- 7. CHIMNEY HEADER ---
z_header = z_mesh_top + 0.78
all_meshes.append(box([1.22, 1.22, 0.012], [0, 0, z_header], COLOR_HEADER_METAL))
all_meshes.append(cyl(0.12, 0.20, [0, 0, z_header + 0.106], COLOR_HEADER_METAL))
all_meshes.append(cyl(0.14, 0.03, [0, 0, z_header + 0.206], COLOR_HEADER_METAL))

for angle in [0, 90, 180, 270]:
    rad = np.radians(angle)
    all_meshes.append(cyl(0.005, 0.08, [0.11 * np.cos(rad), 0.11 * np.sin(rad), z_header + 0.246], COLOR_HEADER_METAL))

all_meshes.append(cone(0.22, 0.08, [0, 0, z_header + 0.306], COLOR_HEADER_METAL))

for bx, by in [(-0.58, -0.58), (-0.58, 0.58), (0.58, -0.58), (0.58, 0.58)]:
    all_meshes.append(box([0.06, 0.06, 0.05], [bx, by, z_header], COLOR_HEADER_METAL))
    kx = bx + (0.065 if bx > 0 else -0.065)
    all_meshes.append(cyl(0.020, 0.02, [kx, by, z_header], COLOR_PLASTIC_DARK))

# --- JOIN AND EXPORT ---
# Concatenate all parts into a single combined mesh geometry
joined_mesh = trimesh.util.concatenate(all_meshes)

# Weld contiguous vertices along touching boundaries
joined_mesh.merge_vertices()

# Export as a single unified mesh object
joined_mesh.export("onion_system_single_mesh.stl")

# Create scene with the merged mesh for visualization
scene = trimesh.Scene(joined_mesh)
scene.export("onion_system_figure_twin.glb")
print("Joined mesh exported successfully!")
scene.show()

Joined mesh exported successfully!
